In [ ]:
# Canonical analysis definitions (single source of truth: analysis/_canonical_defs.py).
# Importing these overrides any local fallback definitions later in the notebook.
# Edits to filters / score-mapping / bucket-rule belong in _canonical_defs.py, not here.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("analysis").resolve()))
from _canonical_defs import (
    ANCHOR, L2_CELLS, AXIS_CELLS, VERDICT_SCORE,
    candidate_score, sign,
    any_tier_disagreement, boundary_flip, strict_reversal,
    same_side_magnitude_shift, strong_to_weak_shift,
    assign_bucket,
)


# 10b — Variance partition without F-axis

*Renamed from `06b_variance_without_F.ipynb` 2026-05-25.*

Robustness check for §5 framing: does the framework hold without the F-axis (CoT vs label-only)?

**Why:** F-axis is vulnerable to the "you handicapped one judge" critique (F2 = label-only is plausibly degraded). C, D, and L1 are immune to that critique. This notebook recomputes the headline metrics restricted to F=F1 cells + L1 swap, all of which keep the judge in full reasoning configuration.

**Inputs:** `results/verdict_scores_wide.csv` from notebook 06. No new judge calls.

**Outputs:**
- `results/dropF_bucket_assignments.csv` — buckets computed on F1-only cells + L1
- `results/dropF_strict_matrix.csv` — strict reversal matrix restricted to F1+L1
- `results/dropF_metrics.json` — headline numbers for §5

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

results_dir = Path('results')
wide = pd.read_csv(results_dir / 'verdict_scores_wide.csv', index_col='uid')

F1_CELLS = ['C1F1D1', 'C1F1D2', 'C2F1D1', 'C2F1D2']
F2_CELLS = ['C1F2D1', 'C1F2D2', 'C2F2D1', 'C2F2D2']
ALL_L2 = F1_CELLS + F2_CELLS

F1_PLUS_L1 = F1_CELLS + (['L1_claude'] if 'L1_claude' in wide.columns else [])

print(f'Loaded {wide.shape[0]} prompts × {wide.shape[1]} cells')
print(f'F1 subset (no degradation): {F1_CELLS}')
print(f'F1 + L1 subset (immune-to-critique cells): {F1_PLUS_L1}')

## 1. Bucket reassignment dropping F-axis

Reuses the same B1/B2/B3 rule as nb 06. Now applied to:
- F1-only (4 cells: C and D vary, F held at F1)
- F1 + L1_claude (5 cells: adds the cross-vendor judge swap)
- F2-only (4 cells, sanity check on the other side)
- All 8 L2 (anchor full-set comparison)

In [ ]:
def assign_bucket(row):
    signs = np.sign(row.dropna())
    if len(signs) < max(2, len(row) // 2):
        return 'B3_unresolved'
    pos = (signs > 0).sum()
    neg = (signs < 0).sum()
    zero = (signs == 0).sum()
    n = len(signs)
    if pos == n or neg == n or zero == n:
        return 'B1_stable'
    majority = max(pos, neg, zero) / n
    if majority < 0.6 or zero > n * 0.4:
        return 'B3_unresolved'
    return 'B2_sensitive'

subsets = {
    'all_8_L2': ALL_L2,
    'F1_only_4cells': F1_CELLS,
    'F1_plus_L1_5cells': F1_PLUS_L1,
    'F2_only_4cells': F2_CELLS,
}

bucket_summary = {}
for name, cells in subsets.items():
    avail = [c for c in cells if c in wide.columns]
    if not avail:
        continue
    buckets = wide[avail].apply(assign_bucket, axis=1)
    props = (buckets.value_counts(normalize=True) * 100).round(1).to_dict()
    bucket_summary[name] = {'cells': avail, 'props': props}
    print(f'{name} ({len(avail)} cells):')
    for b in ['B1_stable', 'B2_sensitive', 'B3_unresolved']:
        print(f'  {b}: {props.get(b, 0):.1f}%')
    print()

# Save the F1+L1 buckets (the headline subset)
headline_buckets = wide[F1_PLUS_L1].apply(assign_bucket, axis=1).rename('bucket')
headline_buckets.to_csv(results_dir / 'dropF_bucket_assignments.csv')
print(f'Saved dropF_bucket_assignments.csv ({len(headline_buckets)} prompts)')

## 2. Strict reversal matrix restricted to F1 + L1

Pairwise strict-reversal rate (excludes ties) on the immune-to-critique cells only. Compare to the full 9-cell version in `results/flip_matrix.csv` (loose) and the strict version in nb 06.

In [ ]:
def strict_flip_rate(s1, s2):
    common = pd.DataFrame({'a': s1, 'b': s2}).dropna()
    common = common[(common['a'] != 0) & (common['b'] != 0)]
    if len(common) == 0:
        return np.nan, 0
    sign_a = np.sign(common['a'])
    sign_b = np.sign(common['b'])
    return (sign_a != sign_b).mean() * 100, len(common)

cells = F1_PLUS_L1
strict_dropF = pd.DataFrame(index=cells, columns=cells, dtype=float)
for c1, c2 in combinations(cells, 2):
    rate, _ = strict_flip_rate(wide[c1], wide[c2])
    strict_dropF.loc[c1, c2] = rate
    strict_dropF.loc[c2, c1] = rate

print('Strict reversal matrix (F1 + L1, immune-to-critique cells):')
print(strict_dropF.round(1))

strict_dropF.to_csv(results_dir / 'dropF_strict_matrix.csv')

# Headline ranges per axis-type
C_pairs = [('C1F1D1', 'C2F1D1'), ('C1F1D2', 'C2F1D2')]
D_pairs = [('C1F1D1', 'C1F1D2'), ('C2F1D1', 'C2F1D2')]
L1_pairs = [('L1_claude', c) for c in F1_CELLS] if 'L1_claude' in cells else []

def range_of(pairs):
    vals = [strict_dropF.loc[a, b] for a, b in pairs if not pd.isna(strict_dropF.loc[a, b])]
    return (min(vals), max(vals)) if vals else (np.nan, np.nan)

C_lo, C_hi = range_of(C_pairs)
D_lo, D_hi = range_of(D_pairs)
L1_lo, L1_hi = range_of(L1_pairs)

print(f'\nStrict-reversal ranges (F1-only, no degradation):')
print(f'  C-axis (rubric): {C_lo:.1f}–{C_hi:.1f}%')
print(f'  D-axis (scope):  {D_lo:.1f}–{D_hi:.1f}%')
print(f'  L1 (judge swap): {L1_lo:.1f}–{L1_hi:.1f}%')

## 3. Per-prompt variance contribution from C, D, L1

For each prompt, compute the standard deviation of its mean_score across the F1+L1 subset. Per-prompt SD captures spec-induced variance for that prompt holding F at F1 (no degradation). Aggregate distribution → how much variance is in the data when F is excluded.

In [ ]:
subset_scores = wide[F1_PLUS_L1]
per_prompt_sd = subset_scores.std(axis=1, ddof=0)
per_prompt_range = subset_scores.max(axis=1) - subset_scores.min(axis=1)

print(f'Per-prompt variance across {len(F1_PLUS_L1)} F1+L1 cells (F-axis dropped):')
print(f'  Mean SD across prompts: {per_prompt_sd.mean():.3f}')
print(f'  Median SD: {per_prompt_sd.median():.3f}')
print(f'  % prompts with SD > 0.5 (substantial spread): {100 * (per_prompt_sd > 0.5).mean():.1f}%')
print(f'  % prompts with range ≥ 2 (verdict swing across half the scale): {100 * (per_prompt_range >= 2).mean():.1f}%')
print(f'  % prompts with range ≥ 1 (any non-trivial spread): {100 * (per_prompt_range >= 1).mean():.1f}%')

# Compare to the all-8-L2 version for context
all_scores = wide[ALL_L2]
all_sd = all_scores.std(axis=1, ddof=0)
all_range = all_scores.max(axis=1) - all_scores.min(axis=1)

print(f'\nCompare to all 8 L2 cells (F included):')
print(f'  Mean SD: {all_sd.mean():.3f}')
print(f'  % SD > 0.5: {100 * (all_sd > 0.5).mean():.1f}%')
print(f'  % range ≥ 2: {100 * (all_range >= 2).mean():.1f}%')

## 4. Bootstrap CI on dropped-F bucket proportions

Same protocol as nb 06: resample 500 prompts with replacement, recompute bucket proportions on the F1+L1 subset, 1000 iterations.

In [ ]:
def bootstrap_buckets(subset_wide, n_iter=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(subset_wide)
    proportions = []
    for _ in range(n_iter):
        sampled = subset_wide.sample(n=n, replace=True, random_state=rng.integers(0, 1_000_000))
        b = sampled.apply(assign_bucket, axis=1)
        props = b.value_counts(normalize=True).to_dict()
        proportions.append({k: props.get(k, 0) for k in ['B1_stable', 'B2_sensitive', 'B3_unresolved']})
    return pd.DataFrame(proportions)

boot = bootstrap_buckets(wide[F1_PLUS_L1], n_iter=1000)

boot_summary = {}
print('Bootstrap (1000 iter) on F1+L1 subset — mean [95% CI]:')
for col in ['B1_stable', 'B2_sensitive', 'B3_unresolved']:
    mean = boot[col].mean() * 100
    lo, hi = np.percentile(boot[col] * 100, [2.5, 97.5])
    boot_summary[col] = [mean, lo, hi]
    print(f'  {col}: {mean:.1f}% [{lo:.1f}%, {hi:.1f}%]')

## 5. Save metrics for §5

In [7]:
metrics = {
    'n_prompts': int(len(wide)),
    'F1_only_cells': F1_CELLS,
    'F1_plus_L1_cells': F1_PLUS_L1,
    'bucket_summary_by_subset': bucket_summary,
    'bucket_bootstrap_F1_plus_L1_95ci': boot_summary,
    'strict_reversal_ranges_F1_only': {
        'C_axis': [float(C_lo), float(C_hi)],
        'D_axis': [float(D_lo), float(D_hi)],
        'L1_swap': [float(L1_lo), float(L1_hi)] if not pd.isna(L1_lo) else None,
    },
    'per_prompt_variance_F1_plus_L1': {
        'mean_sd': float(per_prompt_sd.mean()),
        'median_sd': float(per_prompt_sd.median()),
        'pct_sd_above_0_5': float(100 * (per_prompt_sd > 0.5).mean()),
        'pct_range_above_2': float(100 * (per_prompt_range >= 2).mean()),
        'pct_range_above_1': float(100 * (per_prompt_range >= 1).mean()),
    },
    'per_prompt_variance_all_8_L2': {
        'mean_sd': float(all_sd.mean()),
        'pct_sd_above_0_5': float(100 * (all_sd > 0.5).mean()),
        'pct_range_above_2': float(100 * (all_range >= 2).mean()),
    },
}

with open(results_dir / 'dropF_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved:')
print('  results/dropF_bucket_assignments.csv')
print('  results/dropF_strict_matrix.csv')
print('  results/dropF_metrics.json')
print()
print('--- Headline number for §5 ---')
f1l1 = bucket_summary['F1_plus_L1_5cells']['props']
non_b1_pct = 100 - f1l1.get('B1_stable', 0)
print(f'Even excluding F-axis (label-only), {non_b1_pct:.1f}% of prompts are B2 or B3')
print(f'across reasonable specification choices that no engineer can call degraded')
print(f'(rubric content + scope declaration + cross-vendor judge swap).')

Saved:
  results/dropF_bucket_assignments.csv
  results/dropF_strict_matrix.csv
  results/dropF_metrics.json

--- Headline number for §5 ---
Even excluding F-axis (label-only), 54.8% of prompts are B2 or B3
across reasonable specification choices that no engineer can call degraded
(rubric content + scope declaration + cross-vendor judge swap).
